In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **CREATE FLAG PARAMETERS**

In [0]:
dbutils.widgets.text("inremental_flag","0")

In [0]:
incremental_flag = dbutils.widgets.get("inremental_flag")

In [0]:
print(incremental_flag)

In [0]:
%sql
select * from parquet.`abfss://silver@carsaledatalake.dfs.core.windows.net/CarSales`

Creating Dimension Dealer

Fetching Relative Columns

In [0]:
df_src = spark.sql('''select distinct(Dealer_ID)as Dealer_ID, DealerName from parquet.`abfss://silver@carsaledatalake.dfs.core.windows.net/CarSales`''')

In [0]:
df_src.display()

Dim_Dealer Sink Initial and Incremental (bring schema if table is not exists)

In [0]:
if(spark.catalog.tableExists('cars_catalog.gold.dim_dealer')):
    df_sink=spark.sql('''select dim_dealer_key, Dealer_ID, DealerName from cars_catalog.gold.dim_dealer''')
else:
    df_sink = spark.sql('''select 1 as dim_dealer_key, Dealer_ID, DealerName from parquet.`abfss://silver@carsaledatalake.dfs.core.windows.net/CarSales` where 1=0''')


In [0]:
df_sink.display()

Filtering New Records and Old Records

In [0]:
df_filter = df_src.join(df_sink,df_src.Dealer_ID == df_sink.Dealer_ID,'left').select(df_src['Dealer_ID'],df_src['DealerName'],df_sink['dim_dealer_key'])

In [0]:
df_filter.show()

### > ### - Filter OLD RECORDS

In [0]:
from pyspark.sql.functions import col
df_filter_old = df_filter.filter(col('dim_Dealer_key').isNotNull())


In [0]:
df_filter_old.show()

Filter New Records

In [0]:
df_filter_new = df_filter.filter(col("dim_Dealer_key").isNull()).select(col("Dealer_ID"),col('DealerName'))

In [0]:
df_filter_new.display()

**Create** **Surrogate** **Key**

**Fetch the MAX Surrogate key from exisiting table**

In [0]:
#fetch max surrogate key from existing table
if(incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql(''' select max(dim_dealer_key) from cars_catalog.gold.dim_dealer''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key Column and ADD Max Surrogate Key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_dealer_key', max_value + monotonically_increasing_id())

In [0]:
df_filter_new.display()

### Create final_df = df_filter_old + df_filter_new

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.display()

### SCD TYPE - 1(upsert)

In [0]:
from delta.tables import DeltaTable

In [0]:
#incremental run
if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    # creating DeltaTable object
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@carsaledatalake.dfs.core.windows.net/dim_dealer")
    # now apply merge statement
    # alias are used to telling it is a our target table(it is more readable)
    delta_tbl.alias('trg').merge(df_final.alias('src'), 'trg.dim_dealer_key = src.dim_dealer_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    # we have many ways to do this for dimension table. BUT this is a standard solution for implementing updating all columns in dimension table

    
    #initial run
else:
    df_final.write.format("delta").mode("overwrite").option("path", "abfss://gold@carsaledatalake.dfs.core.windows.net/dim_dealer").saveAsTable("cars_catalog.gold.dim_dealer")

In [0]:
%sql
select * from cars_catalog.gold.dim_dealer